# Calculus and Optimization for Machine Learning: Interactive Laboratory

Training machine learning models is fundamentally an optimization problem: we define a parameterized model family, evaluate a scalar loss function assessing empirical discrepancy on observed data, and calculate gradients with respect to parameters to iteratively step toward optimal configurations.

### Table of Contents
1. [Derivatives & Numerical Verification](#1-derivatives--numerical-verification)
2. [Reverse-Mode Autodiff Engine (Micro-Autograd)](#2-reverse-mode-autodiff-engine-micro-autograd)
3. [Multivariate Gradients & Loss Surface Contours](#3-multivariate-gradients--loss-surface-contours)
4. [The Jacobian & The Hessian (Curvature Analysis)](#4-the-jacobian--the-hessian-curvature-analysis)
5. [Convexity, Saddle Points, and Ill-Conditioned Surfaces](#5-convexity-saddle-points-and-ill-conditioned-surfaces)
6. [Optimizer Battle: Gradient Descent vs. Momentum vs. Adam](#6-optimizer-battle-gradient-descent-vs-momentum-vs-adam)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("Loaded interactive calculus lab environment.")

## 1. Derivatives & Numerical Verification
The derivative $f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$ is approximated numerically using the symmetric central difference formula, which has truncation error $\mathcal{O}(h^2)$:
$$f'(x) \approx \frac{f(x + h) - f(x - h)}{2h}$$

In [ ]:
# Target function: f(x) = x * exp(-x^2)
# Analytical derivative via product & chain rule: f'(x) = (1 - 2*x^2) * exp(-x^2)
f = lambda x: x * np.exp(-(x ** 2))
f_prime_analytic = lambda x: (1.0 - 2.0 * (x ** 2)) * np.exp(-(x ** 2))

x_vals = np.linspace(-2.5, 2.5, 100)
h = 1e-5
f_prime_num = (f(x_vals + h) - f(x_vals - h)) / (2 * h)

max_diff = np.max(np.abs(f_prime_num - f_prime_analytic(x_vals)))
print(f"Maximum absolute error of central difference: {max_diff:.2e}")
assert max_diff < 1e-8

## 2. Reverse-Mode Autodiff Engine (Micro-Autograd)
Deep learning models compute gradients using Reverse-Mode Automatic Differentiation (backpropagation). Intermediate nodes track their operands in a directed acyclic graph (DAG), executing the multivariate chain rule in reverse topological order.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev:
                    build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for n in reversed(topo):
            n._backward()

# Evaluate: L = (w1 * x1) + (w2 * x2)
w1, x1 = Value(2.0), Value(-3.0)
w2, x2 = Value(-1.5), Value(4.0)
L = (w1 * x1) + (w2 * x2)
L.backward()

print(f"Loss L = {L.data:.2f}")
print(f"dL/dw1 = {w1.grad:.2f} (Expected x1 = -3.0)")
print(f"dL/dw2 = {w2.grad:.2f} (Expected x2 = 4.0)")

## 3. Multivariate Gradients & Loss Surface Contours
The gradient $\nabla f(\mathbf{x}) = \left[ \frac{\partial f}{\partial x_1}, \dots, \frac{\partial f}{\partial x_N} \right]^T$ points in the direction of greatest instantaneous increase. Stepping in direction $-\nabla f$ minimizes the loss.

In [ ]:
# Visualizing gradients on an anisotropic quadratic loss bowl
# f(x, y) = 0.5 * (x^2 + 5*y^2)
f_loss = lambda x, y: 0.5 * (x ** 2 + 5.0 * (y ** 2))

x_grid = np.linspace(-4, 4, 30)
y_grid = np.linspace(-2, 2, 30)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f_loss(X, Y)

# Analytical gradients
U = X
V = 5.0 * Y

plt.figure(figsize=(7, 5))
plt.contour(X, Y, Z, levels=15, cmap="viridis")
plt.quiver(X[::2, ::2], Y[::2, ::2], -U[::2, ::2], -V[::2, ::2], color="red", scale=40, label="Negative Gradient (-grad)")
plt.title("Loss Surface Contours & Descent Directions", fontweight="bold")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.tight_layout()
plt.show()

## 4. The Jacobian & The Hessian (Curvature Analysis)
- **Jacobian** $J \in \mathbb{R}^{M \times N}$: First-order derivatives of vector-valued map $F: \mathbb{R}^N \to \mathbb{R}^M$.
- **Hessian** $H \in \mathbb{R}^{N \times N}$: Second-order partial derivatives describing curvature:
$$H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}$$
Second-order Taylor expansion: $f(\mathbf{x} + \Delta \mathbf{x}) \approx f(\mathbf{x}) + \nabla f^T \Delta \mathbf{x} + \frac{1}{2} \Delta \mathbf{x}^T H \Delta \mathbf{x}$.

In [ ]:
# Hessian analysis at stationary point (0, 0)
# Saddle point: f(x, y) = x^2 - 3*y^2
f_saddle = lambda v: v[0] ** 2 - 3.0 * (v[1] ** 2)

# Numerical Hessian
def get_hessian(fn, pt, h=1e-4):
    n = len(pt)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            p_pp, p_pm, p_mp, p_mm = pt.copy(), pt.copy(), pt.copy(), pt.copy()
            p_pp[i] += h; p_pp[j] += h
            p_pm[i] += h; p_pm[j] -= h
            p_mp[i] -= h; p_mp[j] += h
            p_mm[i] -= h; p_mm[j] -= h
            H[i, j] = (fn(p_pp) - fn(p_pm) - fn(p_mp) + fn(p_mm)) / (4 * h * h)
    return H

H_at_origin = get_hessian(f_saddle, np.array([0.0, 0.0]))
eigenvals = np.linalg.eigvalsh(H_at_origin)
print("Hessian matrix at origin:")
print(np.round(H_at_origin, 4))
print(f"Eigenvalues of Hessian: {np.round(eigenvals, 4)}")
print(f"Stationary Point Classification: {'Saddle Point' if np.any(eigenvals > 0) and np.any(eigenvals < 0) else 'Extremum'}")

## 5. Optimizer Battle: Gradient Descent vs. Momentum vs. Adam
Comparing convergence trajectories on the challenging Rosenbrock valley function:
$$f(x, y) = (1 - x)^2 + 100 (y - x^2)^2$$

In [ ]:
rosen = lambda v: (1.0 - v[0]) ** 2 + 100.0 * (v[1] - v[0] ** 2) ** 2
grad_rosen = lambda v: np.array([
    -2.0 * (1.0 - v[0]) - 400.0 * v[0] * (v[1] - v[0] ** 2),
    200.0 * (v[1] - v[0] ** 2)
])

start = np.array([-1.2, 1.0])
steps = 250

# Optimizer 1: Vanilla GD with learning rate 0.001
p_gd = start.copy()
loss_gd = []
for _ in range(steps):
    p_gd -= 0.001 * grad_rosen(p_gd)
    loss_gd.append(rosen(p_gd))

# Optimizer 2: Adam with learning rate 0.05
p_adam = start.copy()
m, v = np.zeros(2), np.zeros(2)
loss_adam = []
for t in range(1, steps + 1):
    g = grad_rosen(p_adam)
    m = 0.9 * m + 0.1 * g
    v = 0.999 * v + 0.001 * (g ** 2)
    m_hat = m / (1.0 - 0.9 ** t)
    v_hat = v / (1.0 - 0.999 ** t)
    p_adam -= (0.05 / (np.sqrt(v_hat) + 1e-8)) * m_hat
    loss_adam.append(rosen(p_adam))

plt.figure(figsize=(8, 4.5))
plt.plot(loss_gd, label="Standard Gradient Descent", color="#d62728")
plt.plot(loss_adam, label="Adam Optimizer", color="#1f77b4", lw=2)
plt.yscale("log")
plt.title("Convergence Trajectory on Rosenbrock Function (Log Scale)", fontweight="bold")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()